In [1]:
from experiments.dj.posterior_tables import SBVGPConfig
from experiments.dj.sysident_tables import SIConfig
from experiments.dj.result_tables import SBVGPResult2, SIResult, FlowPriorResult
from experiments.dj.dataloader_tables import DataLoaderConfig

from task_transfer.utils.insilico_stimuli import generate_gabors
from task_transfer.ml_lib.data_loading import build_dataloaders

from task_transfer.evaluation.evaluate_generative_model import compute_logl

# import torch
import datajoint as dj
from experiments.dj.dataloader_tables import DataLoaderConfig, AltDataLoaderConfig
from experiments.dj.schema import schema
from experiments.dj.likelihood_tables import LikelihoodConfig

from experiments.dj.prior_tables import FlowPriorConfig, AdaptPriorConfig
from experiments.dj.sysident_tables import SIConfig
from experiments.dj.posterior_tables import SBVGPConfig
from experiments.dj.result_tables import (
    FlowPriorResult,
    FPSamples,
    FPSamplesConfig,
    LikelihoodResult,
    MLPCondSamples,
    MLPCondSamplesConfig,
    SBVGPResult2,
    SIResult,
    AdaptPriorResult,
    AdaptPriorSamplesConfig,
    SBVGPAdaptedResult,
)
from experiments.dj.evaluation_tables import SIEval, SBVGPEval
from experiments.dj.trainer_tables import (
    FPTrainerConfig,
    LLTrainerConfig,
    SBVGPTrainerConfig,
    SITrainerConfig,
    AdaptPriorTrainer,
)
from experiments.dj.dj_helpers import drop_schema_dot_jobs
from experiments.dj.dj_helpers import fetch_best_model_results
from task_transfer.ml_lib.data_loading import build_dataloaders

from collections import OrderedDict
from task_transfer.utils.utils import dict_product
from experiments.dj.evaluation_tables import FlowPriorEval

import torch
import matplotlib.pyplot as plt
import seaborn as sns


torch.manual_seed(42)

[2024-08-24 17:13:32,661][INFO]: Connecting sshrinivasan@134.76.19.44:3306
[2024-08-24 17:13:32,668][INFO]: Connected sshrinivasan@134.76.19.44:3306
/usr/local/lib/python3.8/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [2]:
DataLoaderConfig()

id,data_fname,train_prop,val_prop
05977a317062b759857ee411a2e60648,/src/project/data/synthetic/haefner_2afc/haefner_2neuron_task1.pkl,0.7,0.2
260a5ea8175f75eaef132f42873ad14a,/src/project/data/synthetic/haefner_2afc/original_haefner_2afc_task_1_dataset.pkl,0.7,0.2
4477b5e82704db0bc19727864c7ef5aa,/src/project/data/synthetic/haefner_2afc/haefner_4neuron_task1.pkl,0.7,0.2
94efb58694007205fac996d7963f88c5,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task2_dataset.pkl,0.7,0.2
b8379e7d6998fc94a08a9a3742eec12d,/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl,0.7,0.2
d74090584b0b974c4444a5ec64c3d87d,/src/project/data/synthetic/haefner_2afc/haefner_model_2neuron_highdelta_task2_dataset.pkl,0.7,0.2
f1ae78885d2ace1ba976199d4cf1a4d6,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_task1_dataset.pkl,0.7,0.2
f7b32dd97feda9f34e2b47e24fa3d18b,/src/project/data/synthetic/haefner_2afc/haefner_model_1neuron_highdelta_task1_dataset.pkl,0.7,0.2


In [3]:
SIResult & "dl_id = 'f1ae78885d2ace1ba976199d4cf1a4d6'"

si_id,trainer_id,dl_id,"train_ll_mean mean per dimension, per sample, in nats",train_ll_sem standard error of the mean,val_ll_mean,val_ll_sem,test_ll_mean,test_ll_sem,tracker_output,eval_output,model
6deb07682dd4919b2ebe01c255b23445,a2a0e79d4c785fead2a84d56420b6063,f1ae78885d2ace1ba976199d4cf1a4d6,-0.10329186916351318,0.19080695509910583,-0.138202965259552,0.3346806764602661,-0.13471125066280365,0.46513110399246216,=BLOB=,=BLOB=,=BLOB=
6deb07682dd4919b2ebe01c255b23445,f89651063b51487dcdf4041336ef89db,f1ae78885d2ace1ba976199d4cf1a4d6,-0.4223896563053131,0.1585923731327057,-0.44463881850242615,0.29623329639434814,-0.47488656640052795,0.4288819432258606,=BLOB=,=BLOB=,=BLOB=


In [4]:
(
    SITrainerConfig
    & "(id = 'a2a0e79d4c785fead2a84d56420b6063') or (id = 'f89651063b51487dcdf4041336ef89db')"
)

id,lr,weight_decay,n_epochs,batch_size,early_stopping_threshold,early_stopping_patience
a2a0e79d4c785fead2a84d56420b6063,0.001,0.001,500,128,10,10
f89651063b51487dcdf4041336ef89db,0.001,0.001,250,128,10,10


In [5]:
SBVGPTrainerConfig & "(id = 'a2a0e79d4c785fead2a84d56420b6063') or (id = 'f89651063b51487dcdf4041336ef89db')"

id,lr,weight_decay,n_epochs,batch_size,early_stopping_threshold,early_stopping_patience
a2a0e79d4c785fead2a84d56420b6063,0.001,0.001,500,128,10,10
f89651063b51487dcdf4041336ef89db,0.001,0.001,250,128,10,10


In [6]:
SBVGPResult2 & "dl_id = 'f1ae78885d2ace1ba976199d4cf1a4d6'"

sbvp_id,sbvp_trainer_id,fp_samples_id from FlowPriorResult,dl_id from FlowPriorResult and DataLoaderConfig,trainer_id from FlowPriorResult and FPTrainerConfig,data_seed,n_samples,mlpcond_samples_id from LikelihoodResult,ll_trainer_id from LikelihoodResult and LLTrainerConfig,"train_ll_mean mean per dimension, per sample, in nats",train_ll_sem standard error of the mean,val_ll_mean,val_ll_sem,test_ll_mean,test_ll_sem,"train_ll_mean_sample mean per dimension, per sample, in nats",train_ll_sem_sample standard error of the mean,val_ll_mean_sample,val_ll_sem_sample,test_ll_mean_sample,test_ll_sem_sample,tracker_output,eval_output,model
77941396a735dc6ee02bcdf4b24896e7,a2a0e79d4c785fead2a84d56420b6063,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,10000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,-0.5151174068450928,0.19482865929603577,-0.5156005024909973,0.35322511196136475,-0.5404559969902039,0.5353862047195435,-0.5129338502883911,0.21075938642024994,-0.5447589159011841,0.5029643774032593,-0.556271493434906,0.5096266269683838,=BLOB=,=BLOB=,=BLOB=
77941396a735dc6ee02bcdf4b24896e7,a2a0e79d4c785fead2a84d56420b6063,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,20000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,0.3079972267150879,0.38551440834999084,0.27508264780044556,0.5963305830955505,0.25942787528038025,0.9809063673019409,0.18732477724552155,0.5000922083854675,0.2153177559375763,1.129582166671753,0.12793630361557007,1.681668996810913,=BLOB=,=BLOB=,=BLOB=
77941396a735dc6ee02bcdf4b24896e7,a2a0e79d4c785fead2a84d56420b6063,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,30000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,-0.44677019119262695,0.14651507139205933,-0.45768290758132935,0.26997628808021545,-0.48012083768844604,0.3898387551307678,-0.4943196475505829,0.15588049590587616,-0.49443697929382324,0.24427685141563416,-0.4943374693393707,0.4307861030101776,=BLOB=,=BLOB=,=BLOB=
77941396a735dc6ee02bcdf4b24896e7,a2a0e79d4c785fead2a84d56420b6063,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,50000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,0.33423590660095215,0.3386373221874237,0.3075169026851654,0.5183398127555847,0.28513118624687195,0.8711202144622803,0.14877654612064362,0.3844969570636749,0.2813514471054077,0.414338082075119,0.17438650131225586,0.9818687438964844,=BLOB=,=BLOB=,=BLOB=
77941396a735dc6ee02bcdf4b24896e7,f89651063b51487dcdf4041336ef89db,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,10000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,-0.5151174664497375,0.19482865929603577,-0.5156005024909973,0.35322511196136475,-0.5404559969902039,0.5353862047195435,-0.5129338502883911,0.21075938642024994,-0.5447589159011841,0.5029643774032593,-0.556271493434906,0.5096266269683838,=BLOB=,=BLOB=,=BLOB=
77941396a735dc6ee02bcdf4b24896e7,f89651063b51487dcdf4041336ef89db,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,20000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,0.3192407488822937,0.3691276013851166,0.2860182225704193,0.5686489343643188,0.26377248764038086,0.9377521872520447,0.18661779165267944,0.5134180784225464,0.21436429023742676,1.1472889184951782,0.13618269562721252,1.6921381950378418,=BLOB=,=BLOB=,=BLOB=
77941396a735dc6ee02bcdf4b24896e7,f89651063b51487dcdf4041336ef89db,89c1053a65023b042dc63f7f852bb5b0,f1ae78885d2ace1ba976199d4cf1a4d6,c40a50ce9c77369770dddd5129836477,42,30000,a67b8eaff13e89e7272e90768c2ab280,f89651063b51487dcdf4041336ef89db,-0.4484858810901642,0.1451604813337326,-0.4597557485103607,0.2681189775466919,-0.48303329944610596,0.38682985305786133,-0.49467232823371887,0.15339

In [7]:
restriction = "dl_id = 'f7b32dd97feda9f34e2b47e24fa3d18b'"
best_si = (SIResult & restriction).fetch(
    download_path="/tmp", order_by="val_ll_mean DESC", limit=1, as_dict=True
)[0]
sbv_restriction = restriction + " and n_samples = 10000"
best_sbv = (SBVGPResult2 & sbv_restriction).fetch(
    download_path="/tmp", order_by="val_ll_mean DESC", limit=1, as_dict=True
)[0]

best_si_model = torch.load(best_si["model"], map_location="cpu")
best_sbv_model = torch.load(best_sbv["model"], map_location="cpu")

In [8]:
best_si

{'si_id': '6deb07682dd4919b2ebe01c255b23445',
 'trainer_id': 'a2a0e79d4c785fead2a84d56420b6063',
 'dl_id': 'f7b32dd97feda9f34e2b47e24fa3d18b',
 'train_ll_mean': 0.032259501516819,
 'train_ll_sem': 0.1662023514509201,
 'val_ll_mean': -0.004755752626806498,
 'val_ll_sem': 0.334189236164093,
 'test_ll_mean': -0.035427775233983994,
 'test_ll_sem': 0.45351076126098633,
 'tracker_output': '/tmp/ce63873b5cf853e83d343addbedbbfe2_tracker_output.pkl',
 'eval_output': '/tmp/ce63873b5cf853e83d343addbedbbfe2_eval_output.pkl',
 'model': '/tmp/ce63873b5cf853e83d343addbedbbfe2_model.pt'}

In [9]:
best_sbv

{'sbvp_id': '77941396a735dc6ee02bcdf4b24896e7',
 'sbvp_trainer_id': 'f89651063b51487dcdf4041336ef89db',
 'fp_samples_id': '89c1053a65023b042dc63f7f852bb5b0',
 'dl_id': 'f7b32dd97feda9f34e2b47e24fa3d18b',
 'trainer_id': 'c40a50ce9c77369770dddd5129836477',
 'data_seed': 42,
 'n_samples': 10000,
 'mlpcond_samples_id': 'a67b8eaff13e89e7272e90768c2ab280',
 'll_trainer_id': 'f89651063b51487dcdf4041336ef89db',
 'train_ll_mean': 0.5242013335227966,
 'train_ll_sem': 0.23360812664031982,
 'val_ll_mean': 0.4427397847175598,
 'val_ll_sem': 0.5255548357963562,
 'test_ll_mean': 0.3845190703868866,
 'test_ll_sem': 0.7051239013671875,
 'train_ll_mean_sample': 0.492057204246521,
 'train_ll_sem_sample': 0.3346768617630005,
 'val_ll_mean_sample': 0.42926648259162903,
 'val_ll_sem_sample': 0.6960524916648865,
 'test_ll_mean_sample': 0.45057594776153564,
 'test_ll_sem_sample': 0.5513153672218323,
 'tracker_output': '/tmp/a22d1bd5155513d18f6a944203cf02fb_tracker_output.pkl',
 'eval_output': '/tmp/a22d1bd515

In [13]:
SIConfig()

id,seed,cond_dist,nonneg_transform,n_layers,nonlin,dropout_rate,init_std,kwargs other parameters such as pre_scale_max
39f03eac90b439e7897b6f4a65464417,42,gamma,exp,3,relu,0.0,0.001,=BLOB=
45f442bf0f2c1dcfb984300f172610bf,42,gamma,exp,1,relu,0.0,0.001,=BLOB=
4babfd012e0443ef1ae2139deb4def7b,42,gamma,exp,2,none,0.0,0.001,=BLOB=
5531fc5da506b401bcdb7f43e282f731,42,gamma,exp,3,none,0.0,0.001,=BLOB=
5c533709d4d10462567539f3529e43b4,100,gamma,exp,2,none,0.0,0.001,=BLOB=
6deb07682dd4919b2ebe01c255b23445,100,gamma,exp,3,relu,0.0,0.001,=BLOB=
8352064bedbeed787d946629a6e80649,42,gamma,exp,2,relu,0.0,0.001,=BLOB=
841daafa5d8da9165d3677d09f28bf56,42,gamma,exp,1,none,0.0,0.001,=BLOB=
8c7c772c5d9577f0dbe5adae2487a504,100,gamma,exp,2,relu,0.0,0.001,=BLOB=
9f184ea53fc4c7c50e794aa0441e8dba,100,gamma,exp,1,none,0.0,0.001,=BLOB=


In [10]:
SIConfig & f"id = '{best_si['si_id']}'"

id,seed,cond_dist,nonneg_transform,n_layers,nonlin,dropout_rate,init_std,kwargs other parameters such as pre_scale_max
6deb07682dd4919b2ebe01c255b23445,100,gamma,exp,3,relu,0.0,0.001,=BLOB=


In [12]:
SBVGPConfig & f"id = '{best_sbv['sbvp_id']}'"

id,seed,nonneg_transform,n_layers,nonlin,dropout_rate,init_std,kwargs
77941396a735dc6ee02bcdf4b24896e7,100,exp,3,relu,0.0,0.001,=BLOB=
